# Autocorrelación Post-Burst: T+0 → T+1 (análisis intraday correcto)

**Por qué el análisis con cierres EOD está mal para day trading:**
- PBT1 entra en la **apertura de T+1** y sale **intraday** con trailing stop
- El cierre EOD no es el PnL real — el edge vive en los primeros 30-90 min de T+1
- Necesitamos barras de 1 minuto para medir lo que realmente ocurre

**Fuente de datos:**
- `burst_intraday_cache.db` — 76 pares (T+0, T+1) con 390 barras 1-min cada día
- `finviz_snapshots.db` — burst magnitude (change_pct del día burst)

**Métricas correctas:**
- Entrada: `open[T+1 09:30]` (apertura)
- MFE: max excursión favorable desde entrada (cuánto sube antes de bajar)
- MAE: max excursión adversa (cuánto baja antes de subir)
- Retorno con trailing stop simulado (para ver cuánto captura una estrategia real)
- Gap up overnight: `open[T+1] / close[T+0] - 1` (¿el stock abre arriba?)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

DB_INTRADAY = '/Users/carlos/proyectos/TRADING/1_PROYECTOS_ACTIVOS/CLAUDE/finviz-dashboard/analysis/burst_intraday_cache.db'
DB_FV       = '/Users/carlos/Library/Application Support/finviz-dashboard/finviz_snapshots.db'

print('OK')

In [ ]:
# --- Cargar pares T+0 / T+1 del cache ---
conn = sqlite3.connect(DB_INTRADAY)

# Tickers con exactamente 2 días (T+0 y T+1)
pairs_df = pd.read_sql("""
    SELECT symbol, MIN(date) as t0_date, MAX(date) as t1_date
    FROM fetched
    GROUP BY symbol
    HAVING COUNT(DISTINCT date) = 2
""", conn)

# Cargar todas las barras 1-min
bars = pd.read_sql('SELECT symbol, date, dt, open, high, low, close, volume FROM bars', conn)
conn.close()

print(f"Pares T+0/T+1 disponibles: {len(pairs_df)}")
print(f"Barras totales: {len(bars):,}")
pairs_df.head()

In [ ]:
# --- Cargar burst magnitude desde finviz ---
conn = sqlite3.connect(DB_FV)
fv = pd.read_sql("""
    SELECT substr(timestamp,1,10) as burst_date, ticker,
           MAX(CAST(REPLACE(change_pct,'%','') AS REAL)) as burst_chg_pct
    FROM snapshots
    WHERE category='Top Gainers' AND change_pct != '0.00%' AND change_pct != ''
    GROUP BY burst_date, ticker
""", conn)
conn.close()

# Merge con pares
pairs_df = pairs_df.merge(
    fv.rename(columns={'ticker': 'symbol', 'burst_date': 't0_date'})[['symbol', 't0_date', 'burst_chg_pct']],
    on=['symbol', 't0_date'],
    how='left'
)
print(f"Con burst_chg_pct: {pairs_df['burst_chg_pct'].notna().sum()} / {len(pairs_df)}")
print(f"\nburst_chg_pct stats:")
print(pairs_df['burst_chg_pct'].describe().round(1))

In [ ]:
# --- Calcular métricas intraday ---

def simulate_trailing_stop(bars_t1, entry_price, trail_pct=0.05):
    """Simula trailing stop bar a bar. Retorna precio de salida y bar de salida."""
    stop = entry_price * (1 - trail_pct)
    peak = entry_price
    for i, row in bars_t1.iterrows():
        peak = max(peak, row['high'])
        stop = max(stop, peak * (1 - trail_pct))
        if row['low'] <= stop:
            return stop, row['dt'], (stop / entry_price - 1) * 100
    # Sin stop: salida al cierre
    last = bars_t1.iloc[-1]
    return last['close'], last['dt'], (last['close'] / entry_price - 1) * 100


records = []
for _, pair in pairs_df.iterrows():
    sym  = pair['symbol']
    t0   = pair['t0_date']
    t1   = pair['t1_date']
    
    bars_t0 = bars[(bars['symbol'] == sym) & (bars['date'] == t0)].sort_values('dt')
    bars_t1 = bars[(bars['symbol'] == sym) & (bars['date'] == t1)].sort_values('dt').reset_index(drop=True)
    
    if len(bars_t0) < 10 or len(bars_t1) < 10:
        continue
    
    close_t0   = bars_t0.iloc[-1]['close']
    open_t1    = bars_t1.iloc[0]['open']
    close_t1   = bars_t1.iloc[-1]['close']
    high_t0    = bars_t0['high'].max()
    
    if open_t1 <= 0 or close_t0 <= 0:
        continue
    
    # Gap overnight
    gap_pct = (open_t1 / close_t0 - 1) * 100
    
    # MFE y MAE desde entrada (open T+1)
    entry = open_t1
    mfe = (bars_t1['high'].max() / entry - 1) * 100
    mae = (bars_t1['low'].min() / entry - 1) * 100
    eod_ret = (close_t1 / entry - 1) * 100
    
    # MFE en primeros 30 min (barras 0-29)
    mfe_30 = (bars_t1.iloc[:30]['high'].max() / entry - 1) * 100
    # MFE en primeros 60 min
    mfe_60 = (bars_t1.iloc[:60]['high'].max() / entry - 1) * 100
    
    # Trailing stop 5% simulado
    exit_price_5, exit_bar_5, ret_trail_5 = simulate_trailing_stop(bars_t1, entry, trail_pct=0.05)
    # Trailing stop 7%
    exit_price_7, exit_bar_7, ret_trail_7 = simulate_trailing_stop(bars_t1, entry, trail_pct=0.07)
    # Stop fijo 7%
    stop_fixed_7 = max(mae, -7.0)  # si baja más de 7%, sale a -7
    
    # ¿Cuándo alcanza el MFE? (bar index)
    mfe_bar = bars_t1['high'].idxmax()
    mae_bar = bars_t1['low'].idxmin()
    
    records.append({
        'symbol':        sym,
        't0_date':       t0,
        't1_date':       t1,
        'burst_chg_pct': pair['burst_chg_pct'],
        'close_t0':      close_t0,
        'high_t0':       high_t0,
        'open_t1':       open_t1,
        'close_t1':      close_t1,
        'gap_pct':       gap_pct,
        'mfe_pct':       mfe,
        'mae_pct':       mae,
        'eod_ret_pct':   eod_ret,
        'mfe_30_pct':    mfe_30,
        'mfe_60_pct':    mfe_60,
        'ret_trail_5':   ret_trail_5,
        'ret_trail_7':   ret_trail_7,
        'mfe_bar':       mfe_bar,
        'mae_bar':       mae_bar,
    })

df = pd.DataFrame(records)
print(f"Trades analizados: {len(df)}")
print(f"\nMétricas clave:")
for col in ['gap_pct', 'mfe_pct', 'mae_pct', 'eod_ret_pct', 'ret_trail_5', 'ret_trail_7']:
    d = df[col].dropna()
    wr = (d > 0).mean()
    print(f"  {col:<18}: mean={d.mean():+.2f}%  median={d.median():+.2f}%  WR={wr:.0%}")

## 1. Distribución de MFE, MAE y retornos en T+1

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

metrics = [
    ('gap_pct',       'Gap overnight (%)',              'Gap open T+1 / close T+0'),
    ('mfe_pct',       'MFE T+1 (%)',                   'Max Favorable Excursion (día completo)'),
    ('mae_pct',       'MAE T+1 (%)',                   'Max Adverse Excursion (día completo)'),
    ('mfe_30_pct',    'MFE primeros 30 min (%)',        'Max subida primeros 30 min'),
    ('ret_trail_7',   'Return trailing 7% (%)',         'Salida con trailing stop 7%'),
    ('eod_ret_pct',   'Return EOD desde open T+1 (%)', 'Return open→close T+1 (EOD proxy - INCORRECTO)'),
]

for idx, (col, xlabel, title) in enumerate(metrics):
    ax = axes[idx]
    data = df[col].dropna()
    
    # Clip outliers para visualización
    p2, p98 = data.quantile([0.02, 0.98])
    data_clip = data.clip(p2, p98)
    
    colors = ['#4CAF50' if v >= 0 else '#F44336' for v in data_clip]
    ax.bar(range(len(data_clip)), sorted(data_clip), color=sorted(colors, key=lambda c: c != '#4CAF50'), alpha=0.7)
    
    ax.axhline(0, color='black', linewidth=1)
    ax.axhline(data.mean(), color='navy', linewidth=2, linestyle='--', 
               label=f'μ={data.mean():+.2f}%')
    ax.axhline(data.median(), color='orange', linewidth=1.5, linestyle=':', 
               label=f'med={data.median():+.2f}%')
    
    wr = (data > 0).mean()
    ax.set_title(f'{title}\nWR={wr:.0%}  n={len(data)}', fontsize=10, fontweight='bold')
    ax.set_xlabel(xlabel, fontsize=9)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    if idx == 5:
        ax.set_facecolor('#fff9e6')  # fondo amarillo para el EOD (metodología incorrecta)
        ax.set_title(f'{title}\nWR={wr:.0%}  n={len(data)}  ⚠️ proxy inválido', 
                     fontsize=9, fontweight='bold', color='#8B4513')

plt.suptitle('Distribución de métricas intraday T+1 — entrada en apertura\n'
             '(barras ordenadas de menor a mayor, verde=ganancia, rojo=pérdida)', 
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pbt1_intraday_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: pbt1_intraday_distributions.png')

## 2. ¿El tamaño del burst (T+0) predice el MFE de T+1?

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

y_metrics = [
    ('gap_pct',      'Gap overnight T+1 (%)'),
    ('mfe_pct',      'MFE T+1 día completo (%)'),
    ('mfe_30_pct',   'MFE T+1 primeros 30 min (%)'),
    ('mfe_60_pct',   'MFE T+1 primeros 60 min (%)'),
    ('ret_trail_7',  'Return trailing 7% (%)'),
    ('mae_pct',      'MAE T+1 (riesgo, %)'),
]

sub_df = df[df['burst_chg_pct'].notna()].copy()
# Clip burst outliers
x_p2, x_p98 = sub_df['burst_chg_pct'].quantile([0.02, 0.98])
sub_df = sub_df[(sub_df['burst_chg_pct'] >= x_p2) & (sub_df['burst_chg_pct'] <= x_p98)]

for idx, (ycol, ylabel) in enumerate(y_metrics):
    ax = axes[idx]
    sub = sub_df[['burst_chg_pct', ycol]].dropna()
    y_p2, y_p98 = sub[ycol].quantile([0.02, 0.98])
    sub = sub[(sub[ycol] >= y_p2) & (sub[ycol] <= y_p98)]
    
    x = sub['burst_chg_pct'].values
    y = sub[ycol].values
    
    colors = ['#4CAF50' if v >= 0 else '#F44336' for v in y]
    ax.scatter(x, y, c=colors, s=40, alpha=0.7, edgecolors='none')
    
    if len(x) >= 5:
        slope, intercept, r, p_val, _ = stats.linregress(x, y)
        xline = np.linspace(x.min(), x.max(), 50)
        ax.plot(xline, slope*xline + intercept, 'navy', linewidth=2,
                label=f'r={r:.3f}  p={p_val:.3f}' + (' *' if p_val < 0.05 else ''))
    
    ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_xlabel('Burst T+0 change_pct (%)', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    wr = (y > 0).mean()
    ax.set_title(f'Burst → {ylabel}\nWR={wr:.0%}  n={len(sub)}', fontsize=9, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.25)

plt.suptitle('¿El tamaño del burst (T+0) predice el comportamiento intraday de T+1?\n'
             'Verde=ganancia, Rojo=pérdida', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pbt1_burst_vs_intraday.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Gap overnight: ¿dónde abre T+1 respecto al close T+0?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Scatter: gap_pct vs MFE T+1
ax1 = axes[0]
sub = df[['gap_pct', 'mfe_pct']].dropna()
sub = sub[sub['gap_pct'].between(*sub['gap_pct'].quantile([0.02, 0.98]))]
colors = ['#4CAF50' if v >= 0 else '#F44336' for v in sub['mfe_pct']]
ax1.scatter(sub['gap_pct'], sub['mfe_pct'], c=colors, s=40, alpha=0.7)
if len(sub) >= 5:
    slope, intercept, r, p_val, _ = stats.linregress(sub['gap_pct'], sub['mfe_pct'])
    xline = np.linspace(sub['gap_pct'].min(), sub['gap_pct'].max(), 50)
    ax1.plot(xline, slope*xline + intercept, 'navy', linewidth=2, label=f'r={r:.3f} p={p_val:.3f}')
ax1.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax1.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax1.set_xlabel('Gap overnight (%)', fontsize=11)
ax1.set_ylabel('MFE T+1 (%)', fontsize=11)
ax1.set_title('Gap up → ¿mayor MFE?', fontsize=11, fontweight='bold')
ax1.legend(); ax1.grid(True, alpha=0.3)

# Scatter: gap_pct vs trailing 7% return
ax2 = axes[1]
sub2 = df[['gap_pct', 'ret_trail_7']].dropna()
sub2 = sub2[sub2['gap_pct'].between(*sub2['gap_pct'].quantile([0.02, 0.98]))]
colors2 = ['#4CAF50' if v >= 0 else '#F44336' for v in sub2['ret_trail_7']]
ax2.scatter(sub2['gap_pct'], sub2['ret_trail_7'], c=colors2, s=40, alpha=0.7)
if len(sub2) >= 5:
    slope, intercept, r, p_val, _ = stats.linregress(sub2['gap_pct'], sub2['ret_trail_7'])
    xline = np.linspace(sub2['gap_pct'].min(), sub2['gap_pct'].max(), 50)
    ax2.plot(xline, slope*xline + intercept, 'navy', linewidth=2, label=f'r={r:.3f} p={p_val:.3f}')
ax2.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax2.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax2.set_xlabel('Gap overnight (%)', fontsize=11)
ax2.set_ylabel('Return trailing 7% (%)', fontsize=11)
ax2.set_title('Gap up → ¿mejor captura con trailing stop?', fontsize=11, fontweight='bold')
ax2.legend(); ax2.grid(True, alpha=0.3)

# Histograma de gaps
ax3 = axes[2]
gap_data = df['gap_pct'].dropna()
gap_p2, gap_p98 = gap_data.quantile([0.02, 0.98])
gap_clip = gap_data.clip(gap_p2, gap_p98)
n_pos = (gap_data > 0).sum()
n_neg = (gap_data <= 0).sum()
ax3.hist(gap_clip[gap_clip > 0], bins=15, color='#4CAF50', alpha=0.7, label=f'Gap up: {n_pos} ({n_pos/len(gap_data):.0%})')
ax3.hist(gap_clip[gap_clip <= 0], bins=15, color='#F44336', alpha=0.7, label=f'Gap down: {n_neg} ({n_neg/len(gap_data):.0%})')
ax3.axvline(gap_data.mean(), color='navy', linewidth=2, linestyle='--', label=f'μ={gap_data.mean():+.1f}%')
ax3.axvline(0, color='black', linewidth=1)
ax3.set_xlabel('Gap overnight T+1 (%)', fontsize=11)
ax3.set_ylabel('Count', fontsize=11)
ax3.set_title('Distribución de gaps overnight post-burst', fontsize=11, fontweight='bold')
ax3.legend(); ax3.grid(True, alpha=0.3)

plt.suptitle('Comportamiento del gap overnight en T+1', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pbt1_gap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Perfil intraday T+1: ¿cuándo ocurre el MFE y el MAE?

In [ ]:
# Bar medio por minuto: high y low promedio normalizado desde open
# Para cada ticker, normalizar precio respecto al open de T+1
all_t1_bars = []

for _, pair in df.iterrows():
    sym = pair['symbol']
    t1  = pair['t1_date']
    entry = pair['open_t1']
    if entry <= 0:
        continue
    bt1 = bars[(bars['symbol'] == sym) & (bars['date'] == t1)].sort_values('dt').reset_index(drop=True)
    if len(bt1) < 30:
        continue
    bt1['bar_idx'] = range(len(bt1))
    bt1['high_norm'] = (bt1['high'] / entry - 1) * 100
    bt1['low_norm']  = (bt1['low']  / entry - 1) * 100
    bt1['close_norm'] = (bt1['close'] / entry - 1) * 100
    bt1['symbol'] = sym
    all_t1_bars.append(bt1[['symbol', 'bar_idx', 'dt', 'high_norm', 'low_norm', 'close_norm']])

bars_all = pd.concat(all_t1_bars, ignore_index=True)

# Perfil medio por bar_idx
profile = bars_all.groupby('bar_idx').agg(
    high_mean=('high_norm', 'mean'),
    low_mean=('low_norm', 'mean'),
    close_mean=('close_norm', 'mean'),
    close_median=('close_norm', 'median'),
    high_pct75=('high_norm', lambda x: x.quantile(0.75)),
    low_pct25=('low_norm', lambda x: x.quantile(0.25)),
    count=('close_norm', 'count')
).reset_index()

# También: % de trades con close_norm > 0 por bar (WR acumulado)
wr_by_bar = bars_all.groupby('bar_idx')['close_norm'].apply(lambda x: (x > 0).mean()).reset_index()
wr_by_bar.columns = ['bar_idx', 'wr']
profile = profile.merge(wr_by_bar, on='bar_idx')

print(f"Barras en perfil: {len(profile)}, tickers: {bars_all['symbol'].nunique()}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 11))

x = profile['bar_idx'].values
# Convertir bar_idx a hora aprox (09:30 + bar_idx minutos)
hours = [f"{9 + (30 + i) // 60}:{(30 + i) % 60:02d}" for i in x]
tick_positions = [0, 30, 60, 90, 120, 150, 180, 210, 240, 270, 300, 330, 360, 389]
tick_labels = [hours[i] for i in tick_positions if i < len(hours)]

# Plot 1: Perfil de precio normalizado
ax1 = axes[0]
ax1.fill_between(x, profile['low_pct25'], profile['high_pct75'], 
                 alpha=0.25, color='steelblue', label='P25-P75 range')
ax1.plot(x, profile['close_mean'], color='steelblue', linewidth=2.5, label='Close medio')
ax1.plot(x, profile['close_median'], color='darkorange', linewidth=1.5, 
         linestyle='--', label='Close mediana')
ax1.axhline(0, color='black', linewidth=1)

# Marcar bar 30 y bar 60 (puntos de referencia PBT1)
for bar_ref, label, color in [(30, 'Bar 30\n(L1 trail)', 'green'), (60, 'Bar 60\n(L2/time stop)', 'red')]:
    if bar_ref < len(x):
        ax1.axvline(bar_ref, color=color, linewidth=1.5, linestyle=':', alpha=0.8)
        ax1.text(bar_ref + 1, profile['close_mean'].max() * 0.9, label, 
                fontsize=9, color=color, va='top')

ax1.set_ylabel('Retorno desde open T+1 (%)', fontsize=11)
ax1.set_title(f'Perfil intraday medio T+1 post-burst (n={bars_all["symbol"].nunique()} tickers)\n'
              'Normalizado a open=0%', fontsize=12)
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)
ax1.set_xticks(tick_positions[:len(tick_labels)])
ax1.set_xticklabels(tick_labels, rotation=45, fontsize=8)

# Plot 2: Win Rate por bar (% trades con close > open en ese momento)
ax2 = axes[1]
colors_wr = ['#4CAF50' if v >= 0.5 else '#F44336' for v in profile['wr']]
ax2.bar(x, profile['wr'] * 100, color=colors_wr, alpha=0.7, width=1)
ax2.axhline(50, color='black', linewidth=1, linestyle='--', label='50% (azar)')
ax2.axhline(60, color='green', linewidth=1, linestyle=':', alpha=0.7, label='60%')

for bar_ref, label, color in [(30, 'Bar 30', 'green'), (60, 'Bar 60', 'red')]:
    if bar_ref < len(x):
        ax2.axvline(bar_ref, color=color, linewidth=1.5, linestyle=':', alpha=0.8)

ax2.set_ylabel('Win Rate (%)', fontsize=11)
ax2.set_ylim(0, 100)
ax2.set_title('Win Rate bar a bar: % tickers por encima del open en cada momento', fontsize=12)
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)
ax2.set_xticks(tick_positions[:len(tick_labels)])
ax2.set_xticklabels(tick_labels, rotation=45, fontsize=8)

plt.suptitle('Perfil intraday T+1 — ¿cuándo es mejor entrar/salir?', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pbt1_intraday_profile.png', dpi=150, bbox_inches='tight')
plt.show()
print('Guardado: pbt1_intraday_profile.png')

## 5. Scatter: burst magnitude vs métricas intraday (buckets)

In [ ]:
bins   = [0, 30, 60, 100, np.inf]
labels = ['<30%', '30-60%', '60-100%', '>100%']
df['burst_bucket'] = pd.cut(df['burst_chg_pct'], bins=bins, labels=labels)
palette = {'<30%': '#1565C0', '30-60%': '#2E7D32', '60-100%': '#F57F17', '>100%': '#B71C1C'}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Box plots MFE y trailing 7% por bucket
for ax, col, title in [
    (axes[0], 'mfe_pct',    'MFE T+1 por tamaño de burst'),
    (axes[1], 'ret_trail_7','Return trailing 7% por tamaño de burst'),
]:
    data_by_bucket = [df[df['burst_bucket'] == b][col].dropna().values for b in labels]
    bp = ax.boxplot(data_by_bucket, patch_artist=True, medianprops={'linewidth': 2, 'color': 'white'})
    for patch, label in zip(bp['boxes'], labels):
        patch.set_facecolor(palette[label])
        patch.set_alpha(0.7)
    ax.axhline(0, color='black', linewidth=1, linestyle='--')
    ax.set_xticklabels([f'{b}\n(n={len(df[df["burst_bucket"]==b])})' for b in labels])
    ax.set_ylabel(col, fontsize=11)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('¿Los bursts más grandes tienen mejor MFE en T+1?', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('pbt1_bucket_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Feature analysis: ¿qué predice un MFE alto en T+1?

**Metodología anti-sesgo — leer antes de ejecutar:**

### Hipótesis pre-registradas (declaradas ANTES de mirar resultados)

| # | Feature | Construcción | Hipótesis | Razonamiento |
|---|---|---|---|---|
| H1 | `close_vs_high_t0` | close_T0 / high_T0 | positiva con MFE | Si cierra cerca del máximo, los shorts no recuperaron terreno → momentum más limpio en T+1 |
| H2 | `vol_intensity_t0` | volume_T0 / price_T0 (shares) normalizado | positiva con MFE | Volumen extremo = más participantes atrapados, más cobertura pendiente |
| H3 | `burst_chg_pct` | change% T+0 | positiva débil | Mayor burst = más FOMO pendiente, pero también más distribución ya realizada |
| H4 | `price_t0` | precio cierre T+0 | negativa | Precio bajo → más volatilidad % pero también más ruido → MFE menos predecible |
| H5 | `burst_range_pct` | (high-low)/open T+0 | negativa | Rango muy amplio = mucha distribución ya ocurrida en T+0 → menos residuo para T+1 |

### Reglas de interpretación (también pre-registradas)
- Corrección Bonferroni: α = 0.05 / 5 tests = **0.01** (umbral real de significancia)
- n=76: con este tamaño, solo efectos grandes (|r| > 0.30) son creíbles
- Si una hipótesis se rechaza → reportarla igualmente, no ocultarla
- Un resultado significativo aquí es **exploratorio**, no confirmatorio (muestra pequeña)

In [ ]:
# --- Construir features de T+0 (información disponible ANTES de las 9:30 de T+1) ---

ALPHA_BONFERRONI = 0.05 / 5
TARGET = 'mfe_pct'

hypotheses = [
    ('close_vs_high',   'H1', 'positiva'),
    ('vol_dollar_t0',   'H2', 'positiva'),
    ('burst_chg_pct',   'H3', 'positiva débil'),
    ('price_t0',        'H4', 'negativa'),
    ('burst_range_pct', 'H5', 'negativa'),
]

feature_records = []
for _, pair in df.iterrows():
    sym = pair['symbol']
    t0  = pair['t0_date']
    bt0 = bars[(bars['symbol'] == sym) & (bars['date'] == t0)].sort_values('dt').reset_index(drop=True)
    if len(bt0) < 30:
        continue
    open_t0  = bt0.iloc[0]['open']
    high_t0  = bt0['high'].max()
    low_t0   = bt0['low'].min()
    close_t0 = bt0.iloc[-1]['close']
    vol_t0   = bt0['volume'].sum()
    if open_t0 <= 0 or close_t0 <= 0:
        continue
    last_hour  = bt0[bt0['dt'] >= '15:00']
    first_hour = bt0[bt0['dt'] <= '10:30']
    feature_records.append({
        'symbol':             sym,
        't0_date':            t0,
        'close_vs_high':      close_t0 / high_t0,
        'vol_dollar_t0':      vol_t0 * close_t0,
        'burst_chg_pct':      pair['burst_chg_pct'],
        'price_t0':           close_t0,
        'burst_range_pct':    (high_t0 - low_t0) / open_t0 * 100,
        'momentum_last_hour': (close_t0 / last_hour.iloc[0]['open'] - 1) * 100 if len(last_hour) > 0 else np.nan,
        'momentum_first_hour':(first_hour.iloc[-1]['close'] / open_t0 - 1) * 100 if len(first_hour) > 0 else np.nan,
    })

df_feat = pd.DataFrame(feature_records).merge(
    df[['symbol', 't0_date', 'mfe_pct', 'ret_trail_7', 'mfe_30_pct']],
    on=['symbol', 't0_date'], how='inner'
)
print(f"Dataset con features: {len(df_feat)} trades")

# --- Tests estadísticos ---
print(f"\nVariable objetivo: {TARGET}")
print(f"n={len(df_feat)}  α_Bonferroni={ALPHA_BONFERRONI:.3f}  (solo |r|>0.30 creíble)\n")
print(f"{'Feature':<22} {'H':>3} {'Esperado':>12} {'r':>7} {'p-val':>8} {'Sig?':>6} {'Confirma?':>10}")
print("-" * 72)

results = []
for feat, h_num, expected in hypotheses:
    sub = df_feat[[feat, TARGET]].dropna()
    sub = sub[sub[feat].between(*sub[feat].quantile([0.01, 0.99])) &
              sub[TARGET].between(*sub[TARGET].quantile([0.01, 0.99]))]
    if len(sub) < 10:
        continue
    r, p = stats.pearsonr(sub[feat].values, sub[TARGET].values)
    sig = '***' if p < ALPHA_BONFERRONI else ('~' if p < 0.05 else 'ns')
    confirms = '✓' if (('positiva' in expected and r > 0.10 and p < 0.05) or
                       ('negativa' in expected and r < -0.10 and p < 0.05)) else '✗'
    print(f"{feat:<22} {h_num:>3} {expected:>12} {r:>+7.3f} {p:>8.4f} {sig:>6} {confirms:>10}")
    results.append({'feature': feat, 'h': h_num, 'expected': expected, 'r': r, 'p': p})

print(f"\n*** p<{ALPHA_BONFERRONI:.3f} (Bonferroni)  ~ p<0.05  ns=no significativo")
print(f"⚠️  n={len(df_feat)}: resultados EXPLORATORIOS, no confirmatorios")

In [ ]:
# --- Construir features de T+0 (toda la información disponible ANTES de las 9:30 de T+1) ---

feature_records = []
for _, pair in df.iterrows():
    sym = pair['symbol']
    t0  = pair['t0_date']

    bt0 = bars[(bars['symbol'] == sym) & (bars['date'] == t0)].sort_values('dt').reset_index(drop=True)
    if len(bt0) < 30:
        continue

    open_t0  = bt0.iloc[0]['open']
    high_t0  = bt0['high'].max()
    low_t0   = bt0['low'].min()
    close_t0 = bt0.iloc[-1]['close']
    vol_t0   = bt0['volume'].sum()

    if open_t0 <= 0 or close_t0 <= 0:
        continue

    # H1: ¿cerró cerca del máximo?
    close_vs_high = close_t0 / high_t0  # 1.0 = cerró en el máximo, 0.5 = cerró en la mitad

    # H2: intensidad de volumen (shares / precio = número de shares, normalizado por float proxy)
    # No tenemos float real → usamos volumen en $ como proxy de actividad
    vol_dollar_t0 = vol_t0 * close_t0  # volumen en dólares

    # H3: burst_chg_pct ya está en df
    burst_chg = pair['burst_chg_pct']

    # H4: precio de cierre T+0
    price_t0 = close_t0

    # H5: rango del día / open (amplitud relativa del burst)
    burst_range_pct = (high_t0 - low_t0) / open_t0 * 100

    # Feature adicional: momentum última hora T+0
    # ¿El precio subió o bajó en la última hora antes del cierre?
    last_hour = bt0[bt0['dt'] >= '15:00']
    first_hour = bt0[bt0['dt'] <= '10:30']
    momentum_last_hour = (close_t0 / last_hour.iloc[0]['open'] - 1) * 100 if len(last_hour) > 0 else np.nan
    momentum_first_hour = (first_hour.iloc[-1]['close'] / open_t0 - 1) * 100 if len(first_hour) > 0 else np.nan

    feature_records.append({
        'symbol':             sym,
        't0_date':            t0,
        'close_vs_high':      close_vs_high,       # H1
        'vol_dollar_t0':      vol_dollar_t0,        # H2
        'burst_chg_pct':      burst_chg,            # H3
        'price_t0':           price_t0,             # H4
        'burst_range_pct':    burst_range_pct,      # H5
        'momentum_last_hour': momentum_last_hour,   # extra
        'momentum_first_hour':momentum_first_hour,  # extra
    })

df_feat = pd.DataFrame(feature_records)
df_feat = df_feat.merge(df[['symbol', 't0_date', 'mfe_pct', 'ret_trail_7', 'mfe_30_pct']], on=['symbol', 't0_date'], how='inner')

print(f"Dataset con features: {len(df_feat)} trades")
print(f"\nFeatures construidas:")
for col in ['close_vs_high', 'vol_dollar_t0', 'burst_chg_pct', 'price_t0', 'burst_range_pct']:
    print(f"  {col:<22}: {df_feat[col].notna().sum()} válidos")